In [75]:
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt

from scipy.optimize import minimize

In [111]:
tickers = ['AAPL', 'JPM', 'XOM', 'JNJ', 'BRK-B', 'GLD', 'VNQ']
data = yf.download(tickers, start='2015-01-01', end='2020-01-01', auto_adjust=True)['Close']
data.head()

[*********************100%***********************]  7 of 7 completed


Ticker,AAPL,BRK-B,GLD,JNJ,JPM,VNQ,XOM
Date,,,,,,,
2015-01-02,24.192606,149.169998,114.080002,76.548607,46.274315,52.549873,57.145554
2015-01-05,23.511063,147.000000,115.800003,76.014008,44.837730,52.837467,55.581936
2015-01-06,23.513269,146.839996,117.120003,75.640457,43.675133,53.361572,55.286457
2015-01-07,23.842979,148.880005,116.430000,77.310310,43.741776,54.179657,55.846653
2015-01-08,24.759079,151.369995,115.940002,77.918175,44.719257,54.384201,56.776203


In [121]:
# Drop any pair with correlation above 0.85
data.corr()

Ticker,AAPL,BRK-B,GLD,JNJ,JPM,VNQ,XOM
Ticker,,,,,,,
AAPL,1.000000,0.919003,0.591294,0.810995,0.941932,0.780208,0.069395
BRK-B,0.919003,1.000000,0.576703,0.921023,0.974044,0.745829,0.242345
GLD,0.591294,0.576703,1.000000,0.605207,0.586249,0.790268,0.112182
JNJ,0.810995,0.921023,0.605207,1.000000,0.896178,0.787118,0.356909
JPM,0.941932,0.974044,0.586249,0.896178,1.000000,0.755854,0.149213
VNQ,0.780208,0.745829,0.790268,0.787118,0.755854,1.000000,0.219956
XOM,0.069395,0.242345,0.112182,0.356909,0.149213,0.219956,1.000000


In [122]:
log_return = np.log(data/ data.shift(1)).dropna()
log_return.head()

Ticker,AAPL,BRK-B,GLD,JNJ,JPM,VNQ,XOM
Date,,,,,,,
2015-01-05,-0.028576,-0.014654,0.014965,-0.007008,-0.031537,0.005458,-0.027743
2015-01-06,0.000094,-0.001089,0.011334,-0.004926,-0.026271,0.009870,-0.005330
2015-01-07,0.013925,0.013797,-0.005909,0.021836,0.001525,0.015215,0.010082
2015-01-08,0.037702,0.016586,-0.004217,0.007832,0.022101,0.003768,0.016508
2015-01-09,0.001072,-0.012631,0.011321,-0.013722,-0.017540,0.000470,-0.001411


In [123]:
span = 252  # one trading year, much more stable

# EWMA mean
mu = log_return.ewm(span=span).mean().iloc[-1] * 252

# EWMA covariance — correct extraction
ewm_cov = log_return.ewm(span=span).cov().iloc[-n_assets:]
cov = pd.DataFrame(
    ewm_cov.values,
    index=tickers,
    columns=tickers
).values * 252

In [124]:
n_assets = len(tickers)
n_portfolios = 10000

results = np.zeros((3, n_portfolios))
all_weights = np.zeros((n_portfolios, n_assets))  
for i in range(n_portfolios):
    w = np.random.random(n_assets)
    w /= w.sum()
    
    all_weights[i] = w  # store weights for this portfolio
    
    ret = w @ mu
    std = np.sqrt(w.T @ cov @ w)
    rf = 0.053  # 5.3% annualised
    sharpe = (ret - rf)/ std
    
    results[0, i] = ret
    results[1, i] = std
    results[2, i] = sharpe

In [125]:
def neg_sharpe(w, mu, cov, rf=0.053):
    ret = w @ mu
    std = np.sqrt(w @ cov @ w)
    return -(ret - rf) / std

def portfolio_variance(w, cov):
    return w.T @ cov @ w

def portfolio_return(w, mu):
    return w @ mu

bounds = [(0, 1)] * n_assets

frontier_returns = np.linspace(mu.min(), mu.max(), 100)
frontier_vols = []
for target in frontier_returns:
    constraints = [
        {'type': 'eq', 'fun': lambda w: w.sum() - 1},
        {'type': 'eq', 'fun': lambda w, t=target: portfolio_return(w, mu) - t}
    ]
    result = minimize(portfolio_variance,
                      x0=np.ones(n_assets) / n_assets,
                      args=(cov,),
                      method='SLSQP',
                      bounds=bounds,
                      constraints=constraints)
    frontier_vols.append(np.sqrt(result.fun))

In [126]:
# 3. Exact MVP
mvp_constraints = [{'type': 'eq', 'fun': lambda w: w.sum() - 1}]
mvp_result = minimize(portfolio_variance,
                      x0=np.ones(n_assets) / n_assets,
                      args=(cov,),
                      method='SLSQP',
                      bounds=bounds,
                      constraints=mvp_constraints)
mvp_weights = pd.Series(mvp_result.x, index=tickers)
mvp_ret = portfolio_return(mvp_result.x, mu)
mvp_vol = np.sqrt(portfolio_variance(mvp_result.x, cov))
mvp_sharpe = (mvp_ret - 0.053) / mvp_vol

# 4. Exact Tangency
tan_constraints = [{'type': 'eq', 'fun': lambda w: w.sum() - 1}]
tan_result = minimize(neg_sharpe,
                      x0=np.ones(n_assets) / n_assets,
                      args=(mu, cov),
                      method='SLSQP',
                      bounds=bounds,
                      constraints=tan_constraints)
tan_weights = pd.Series(tan_result.x, index=tickers)
tan_ret = portfolio_return(tan_result.x, mu)
tan_vol = np.sqrt(portfolio_variance(tan_result.x, cov))
tan_sharpe = (tan_ret - 0.053) / tan_vol

In [127]:
print("MVP Weights:\n", mvp_weights.round(4))
print(f"MVP — Return: {mvp_ret:.2%}, Vol: {mvp_vol:.2%}, Sharpe: {mvp_sharpe:.2f}")

print("\nTangency Weights:\n", tan_weights.round(4))
print(f"Tangency — Return: {tan_ret:.2%}, Vol: {tan_vol:.2%}, Sharpe: {tan_sharpe:.2f}")

MVP Weights:
 AAPL     0.0000
JPM      0.1354
XOM      0.4908
JNJ      0.0611
BRK-B    0.1137
GLD      0.1401
VNQ      0.0588
dtype: float64
MVP — Return: 16.37%, Vol: 6.70%, Sharpe: 1.65

Tangency Weights:
 AAPL     0.2368
JPM      0.0000
XOM      0.5088
JNJ      0.0049
BRK-B    0.2495
GLD      0.0000
VNQ      0.0000
dtype: float64
Tangency — Return: 31.40%, Vol: 8.87%, Sharpe: 2.94
